# SourceExecutor end-to-end test (Avalanche)

This notebook:
1) Gets an Odos quote for inputToken -> USDC (userAddr = SourceExecutor)
2) Assembles Odos calldata
3) Approves the input token to SourceExecutor
4) Calls `SourceExecutor.executeSwapAndBurn`

Prereqs:
- Use the `myvenv` kernel.
- `.env` contains `DEPLOYER_PRIVATE_KEY`.
- The deployer wallet holds the input token and AVAX for gas.
- SourceExecutor is already deployed (set address below).

Warning: this will burn USDC and send a CCTP message on mainnet.


In [1]:
import os
import sys
from dataclasses import dataclass
from decimal import Decimal
from pathlib import Path

from dotenv import load_dotenv
from solcx import compile_standard, install_solc
from web3 import Web3

try:
    from eth_abi import decode as abi_decode
except ImportError:  # pragma: no cover
    from eth_abi import decode_abi as abi_decode

CHAIN_ID = 43114  # Avalanche C-Chain

AVALANCHE_RPC_URL = "https://api.avax.network/ext/bc/C/rpc"
SOURCE_EXECUTOR_ADDRESS = "0xeC20cbb6C99e4b8b78cbd81b27Ce0B7aaD222A2d"

# Tokens (Avalanche examples)
INPUT_TOKEN_ADDRESS = "0x152b9d0FdC40C096757F570A51E494bd4b943E50"  # BTC.b
INPUT_TOKEN_DECIMALS = 8
USDC_ADDRESS = "0xB97EF9Ef8734C71904D8002F8b6Bc66Dd9c48a6E"

# Odos router (Avalanche)
ODOS_ROUTER_ADDRESS = "0x0D05a7D3448512B78fa8A9e46c4872C88C4a0D05"

# CCTP params (Base destination)
DESTINATION_DOMAIN = 6  # Base = 6
DESTINATION_MINT_RECIPIENT = ""  # leave empty to use sender
DESTINATION_CALLER = "0x0000000000000000000000000000000000000000"
MAX_FEE = 0
MIN_FINALITY_THRESHOLD = 1000
HOOK_DATA = "0x01"

# INPUT_AMOUNT = "0.00000001"
INPUT_AMOUNT = "0.00008008"



@dataclass
class Token:
    address: str
    chain_id: int
    decimals: int


def find_repo_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / "solidity").exists() and (parent / "backend").exists():
            return parent
    raise FileNotFoundError("Repo root not found")


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Missing env var: {name}")
    return value


def require_address(label: str, value: str) -> str:
    if not value or value == "0x0000000000000000000000000000000000000000":
        raise ValueError(f"Set {label} address")
    return Web3.to_checksum_address(value)


def address_to_bytes32(addr: str) -> bytes:
    return Web3.to_bytes(hexstr=addr).rjust(32, b"\x00")


root = find_repo_root(Path.cwd())
load_dotenv(root / ".env")

rpc_url = AVALANCHE_RPC_URL
private_key = require_env("DEPLOYER_PRIVATE_KEY")
source_executor_address = require_address("SOURCE_EXECUTOR_ADDRESS", SOURCE_EXECUTOR_ADDRESS)
input_token_address = require_address("INPUT_TOKEN_ADDRESS", INPUT_TOKEN_ADDRESS)
odos_router_address = require_address("ODOS_ROUTER_ADDRESS", ODOS_ROUTER_ADDRESS)
usdc_address = require_address("USDC_ADDRESS", USDC_ADDRESS)

In [2]:
w3 = Web3(Web3.HTTPProvider(rpc_url))
if not w3.is_connected():
    raise ConnectionError("Failed to connect to RPC")

account = w3.eth.account.from_key(private_key)
print("Deployer:", account.address)

destination_mint_recipient = (
    Web3.to_checksum_address(DESTINATION_MINT_RECIPIENT)
    if DESTINATION_MINT_RECIPIENT
    else account.address
)
destination_caller = Web3.to_checksum_address(DESTINATION_CALLER)
print("Destination mint recipient:", destination_mint_recipient)

install_solc("0.8.19")
source_path = root / "solidity" / "SourceExecutor.sol"
source = source_path.read_text()

compiled = compile_standard(
    {
        "language": "Solidity",
        "sources": {"SourceExecutor.sol": {"content": source}},
        "settings": {"outputSelection": {"*": {"*": ["abi", "evm.bytecode"]}}},
    },
    solc_version="0.8.19",
)

contract_data = compiled["contracts"]["SourceExecutor.sol"]["SourceExecutor"]
abi = contract_data["abi"]

source_executor = w3.eth.contract(address=source_executor_address, abi=abi)

CUSTOM_ERROR_SELECTORS = {
    "0x" + Web3.keccak(text="ZeroAddress()")[:4].hex(): "ZeroAddress",
    "0x" + Web3.keccak(text="NativeValueNotAllowed()")[:4].hex(): "NativeValueNotAllowed",
    "0x" + Web3.keccak(text="OdosSwapFailed()")[:4].hex(): "OdosSwapFailed",
    "0x" + Web3.keccak(text="UsdcOutZero()")[:4].hex(): "UsdcOutZero",
    "0x" + Web3.keccak(text="NotOwner()")[:4].hex(): "NotOwner",
    "0x" + Web3.keccak(text="InvalidRecipient()")[:4].hex(): "InvalidRecipient",
    "0x" + Web3.keccak(text="RescueFailed()")[:4].hex(): "RescueFailed",
    "0x" + Web3.keccak(text="ERC20CallFailed(address,bytes4)")[:4].hex(): "ERC20CallFailed",
    "0x" + Web3.keccak(text="ERC20CallReturnedFalse(address,bytes4)")[:4].hex(): "ERC20CallReturnedFalse",
}


def decode_revert_data(data_hex: str) -> str | None:
    if not data_hex:
        return None

    if isinstance(data_hex, bytes):
        data = data_hex
    else:
        hex_str = data_hex[2:] if data_hex.startswith("0x") else data_hex
        data = bytes.fromhex(hex_str)

    if len(data) < 4:
        return None

    selector = "0x" + data[:4].hex()

    if selector == "0x08c379a0":
        message = abi_decode(["string"], data[4:])[0]
        return f"Error({message})"

    if selector == "0x4e487b71":
        code = abi_decode(["uint256"], data[4:])[0]
        return f"Panic({hex(code)})"

    if selector in CUSTOM_ERROR_SELECTORS:
        name = CUSTOM_ERROR_SELECTORS[selector]
        if name in {"ERC20CallFailed", "ERC20CallReturnedFalse"} and len(data) >= 4 + 64:
            token, sel = abi_decode(["address", "bytes4"], data[4:])
            return f"{name}(token={token}, selector=0x{sel.hex()})"
        return f"{name}()"

    return f"Unknown custom error selector {selector}"


def extract_revert_data(err: Exception) -> str | None:
    data = getattr(err, "data", None)
    if isinstance(data, dict):
        data = data.get("data") or data.get("result")
    if isinstance(data, str) and data.startswith("0x"):
        return data

    for arg in getattr(err, "args", []):
        if isinstance(arg, str) and arg.startswith("0x"):
            return arg
        if isinstance(arg, dict) and "data" in arg:
            return arg.get("data")

    return None


def explain_revert(err: Exception) -> str | None:
    data = extract_revert_data(err)
    if not data:
        return None
    return decode_revert_data(data)


Deployer: 0x2299fFB140C4f437E0015fdd9e1956feb2471eA6
Destination mint recipient: 0x2299fFB140C4f437E0015fdd9e1956feb2471eA6


In [3]:
sys.path.insert(0, str(root / "backend"))
from libs import provider

token_in = Token(address=input_token_address, chain_id=CHAIN_ID, decimals=INPUT_TOKEN_DECIMALS)
token_out = Token(address=usdc_address, chain_id=CHAIN_ID, decimals=6)

amount_in = Decimal(INPUT_AMOUNT)
quote = provider.get_quote(
    amount_in=amount_in,
    token_in=token_in,
    token_out=token_out,
    wallet_address=source_executor_address,
)
if quote is None:
    raise RuntimeError("Odos quote failed")

assembled = provider.assemble(quote=quote, wallet_address=source_executor_address)
if assembled is None:
    raise RuntimeError("Odos assemble failed")

odos_data = assembled["transaction"]["data"]
odos_value = int(assembled["transaction"].get("value", 0))
if odos_value != 0:
    raise ValueError("Odos transaction requires native value; contract expects ERC20 swaps only")

amount_in_wei = int(amount_in * (10**token_in.decimals))
print("amount_in_wei:", amount_in_wei)


amount_in_wei: 1


In [4]:
ERC20_ABI = [
    {
        "name": "approve",
        "type": "function",
        "stateMutability": "nonpayable",
        "inputs": [
            {"name": "spender", "type": "address"},
            {"name": "amount", "type": "uint256"},
        ],
        "outputs": [{"name": "", "type": "bool"}],
    },
    {
        "name": "allowance",
        "type": "function",
        "stateMutability": "view",
        "inputs": [
            {"name": "owner", "type": "address"},
            {"name": "spender", "type": "address"},
        ],
        "outputs": [{"name": "", "type": "uint256"}],
    },
    {
        "name": "balanceOf",
        "type": "function",
        "stateMutability": "view",
        "inputs": [{"name": "account", "type": "address"}],
        "outputs": [{"name": "", "type": "uint256"}],
    },
]

input_token = w3.eth.contract(address=token_in.address, abi=ERC20_ABI)

balance = input_token.functions.balanceOf(account.address).call()
if balance < amount_in_wei:
    raise ValueError("Insufficient input token balance")

nonce = w3.eth.get_transaction_count(account.address)

allowance = input_token.functions.allowance(account.address, source_executor_address).call()
if allowance < amount_in_wei:
    latest = w3.eth.get_block("latest")
    max_priority = w3.eth.max_priority_fee
    max_fee = latest["baseFeePerGas"] * 2 + max_priority  # increase if needed

    approve_tx = input_token.functions.approve(source_executor_address, amount_in_wei).build_transaction(
        {
            "from": account.address,
            "nonce": nonce,
            "gas": 120000,
            "chainId": CHAIN_ID,
            "maxFeePerGas": max_fee,
            "maxPriorityFeePerGas": max_priority,
        }
    )

    signed_approve = account.sign_transaction(approve_tx)
    approve_hash = w3.eth.send_raw_transaction(signed_approve.raw_transaction)
    print("Approve tx:", approve_hash.hex())

    approve_receipt = w3.eth.wait_for_transaction_receipt(approve_hash)
    print("Approve status:", approve_receipt.status)

    nonce += 1
else:
    print("Allowance already sufficient; skipping approve")


Allowance already sufficient; skipping approve


In [5]:
hook_data = Web3.to_bytes(hexstr=HOOK_DATA)

latest = w3.eth.get_block("latest")
max_priority = w3.eth.max_priority_fee
max_fee = latest["baseFeePerGas"] * 2 + max_priority  # increase if needed

call_args = (
    token_in.address,
    amount_in_wei,
    odos_data,
    DESTINATION_DOMAIN,
    address_to_bytes32(destination_mint_recipient),
    address_to_bytes32(destination_caller),
    MAX_FEE,
    MIN_FINALITY_THRESHOLD,
    hook_data,
)

tx_params = {
    "from": account.address,
    "nonce": nonce,
    "chainId": CHAIN_ID,
    "value": 0,
    "maxFeePerGas": max_fee,
    "maxPriorityFeePerGas": max_priority,
}

try:
    gas_estimate = source_executor.functions.executeSwapAndBurn(*call_args).estimate_gas(
        {"from": account.address}
    )
except Exception as err:
    decoded = explain_revert(err)
    if decoded:
        print("Decoded revert:", decoded)
    raise

tx_params["gas"] = int(gas_estimate * 1.2)

tx = source_executor.functions.executeSwapAndBurn(*call_args).build_transaction(tx_params)

signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
print("Swap+burn tx:", tx_hash.hex())

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Status:", receipt.status)

try:
    events = source_executor.events.SwapAndBurnExecuted().process_receipt(receipt)
except AttributeError:
    events = source_executor.events.SwapAndBurnExecuted().processReceipt(receipt)

if not events:
    print("No SwapAndBurnExecuted events found")
else:
    for evt in events:
        args = evt["args"]
        print("SwapAndBurnExecuted:", dict(args))


Decoded revert: OdosSwapFailed()


ContractCustomError: ('0x45d4013e', '0x45d4013e')